# Phase 3 Final Inverse PINN

Final clean 13x2 sensor layout with the auxiliary neural-network Bi model. Run the notebook from top to bottom in the Phase3 submission folder. It solves the FDM reference, trains the inverse PINN, plots the final field/Bi recovery, and writes all outputs to `./Phase3_Final_Output/`.

In [ ]:
import csv
import json
import random
import time
from pathlib import Path


import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython.display import display
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve
from torch.optim.lr_scheduler import CosineAnnealingLR

SEED = 2026
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64)

if "__vsc_ipynb_file__" in globals():
    SUBMISSION_DIR = Path(__vsc_ipynb_file__).resolve().parent
else:
    SUBMISSION_DIR = Path.cwd().resolve()
OUTDIR = SUBMISSION_DIR / "Phase3_Final_Output"
FDM_N = 481

RUN_NAME = "final_clean_13x2_auxiliary_nn"
SENSOR_X = [0.02, 0.12, 0.24, 0.34, 0.40, 0.46, 0.50, 0.54, 0.60, 0.66, 0.76, 0.88, 0.98]
SENSOR_Y = [0.98, 1.00]

N_DOMAIN = 3000
N_BOUNDARY = 400
HIDDEN_LAYERS = 4
NEURONS = 64

AUX_BI_HIDDEN_WIDTH = 16
AUX_BI_HIDDEN_LAYERS = 2
H_MIN_GUESS = 500.0
H_MAX_GUESS = 2000.0

W_PDE = 1.0
W_BOTTOM = 10.0
W_TOP = 50.0
W_SIDE = 10.0
W_DATA = 100.0
W_THETA_Y_DATA = 10.0

N_ADAM = 15000
PRINT_EVERY = 500
PINN_LR = 1.0e-3
BI_LR = 1.0e-3
ADAM_ETA_MIN = 1.0e-5
RUN_LBFGS = True
LBFGS_STEPS = 600

print(f"device: {DEVICE}")
print(f"submission directory: {SUBMISSION_DIR}")
print(f"output: {OUTDIR}")
print(f"FDM grid: {FDM_N} x {FDM_N}")
print(f"sensor count: {len(SENSOR_X) * len(SENSOR_Y)}")

In [ ]:
L = 20e-3
D_JET = 2e-3
X_JET = 0.5
K_S = 50.0
QDOT = 1.0e7
T_B = 100.0
T_INF = 25.0
H_SIDE = 25.0

Q_STAR = QDOT * L**2 / (K_S * (T_B - T_INF))
BI_SIDE = H_SIDE * L / K_S
R_OVER_D_DATA = np.array([0.00, 0.25, 0.50, 0.75, 1.05, 1.25, 1.50, 1.75, 1.94, 2.25, 2.50, 2.75, 3.00, 3.25, 3.50, 3.75, 4.00, 4.25, 4.50, 4.75, 5.00, 5.05], dtype=float)
H_DATA = np.array([2070, 1950, 1710, 1490, 1370, 1410, 1500, 1570, 1600, 1540, 1450, 1360, 1260, 1160, 1060, 970, 890, 815, 750, 700, 670, 665], dtype=float)
BI_INTERP = PchipInterpolator(R_OVER_D_DATA, H_DATA * L / K_S, extrapolate=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def true_bi_numpy(x_star):
    r_over_d = np.abs(np.asarray(x_star) - X_JET) * L / D_JET
    return BI_INTERP(r_over_d)

def to_temperature(theta):
    return T_INF + theta * (T_B - T_INF)

x_preview = np.linspace(0.0, 1.0, 501)
fig, ax = plt.subplots(figsize=(6.4, 3.6), dpi=140)
ax.plot(x_preview, true_bi_numpy(x_preview), "k", lw=2)
ax.set_xlabel("x*")
ax.set_ylabel("true Bi_top(x*)")
ax.grid(True, alpha=0.3)
ax.set_title("Empirical top-wall Bi profile")
display(fig)
plt.close(fig)

print(f"Q* = {Q_STAR:.6f}")
print(f"Bi_side = {BI_SIDE:.6f}")
print(f"Bi_top range = {true_bi_numpy(x_preview).min():.4f} to {true_bi_numpy(x_preview).max():.4f}")

## 1. FDM Reference

The notebook solves the reference field directly, rather than loading an external FDM file. This keeps the final submission consistent with the Phase 2 style.

In [ ]:
def node(i, j, n):
    return j * n + i

print(f"Solving FDM reference on {FDM_N} x {FDM_N} grid...")
t0 = time.perf_counter()
xs = np.linspace(0.0, 1.0, FDM_N)
ys = np.linspace(0.0, 1.0, FDM_N)
d = xs[1] - xs[0]
A = lil_matrix((FDM_N * FDM_N, FDM_N * FDM_N))
b = np.zeros(FDM_N * FDM_N)

for j in range(FDM_N):
    for i, x in enumerate(xs):
        row = node(i, j, FDM_N)
        if j == 0:
            A[row, row] = 1.0
            b[row] = 1.0
        elif j == FDM_N - 1:
            A[row, row] = 3.0 + 2.0 * d * float(true_bi_numpy(x))
            A[row, node(i, j - 1, FDM_N)] = -4.0
            A[row, node(i, j - 2, FDM_N)] = 1.0
        elif i == 0:
            A[row, row] = -3.0 - 2.0 * d * BI_SIDE
            A[row, node(i + 1, j, FDM_N)] = 4.0
            A[row, node(i + 2, j, FDM_N)] = -1.0
        elif i == FDM_N - 1:
            A[row, row] = 3.0 + 2.0 * d * BI_SIDE
            A[row, node(i - 1, j, FDM_N)] = -4.0
            A[row, node(i - 2, j, FDM_N)] = 1.0
        else:
            A[row, row] = 4.0
            A[row, node(i - 1, j, FDM_N)] = -1.0
            A[row, node(i + 1, j, FDM_N)] = -1.0
            A[row, node(i, j - 1, FDM_N)] = -1.0
            A[row, node(i, j + 1, FDM_N)] = -1.0
            b[row] = Q_STAR * d**2

theta_ref = spsolve(A.tocsr(), b).reshape(FDM_N, FDM_N)
ref = {"source": f"FDM solved inside Phase3 notebook, n={FDM_N}", "xs": xs, "ys": ys, "theta": theta_ref}
print(f"FDM solved in {time.perf_counter() - t0:.1f} s")
print(f"theta_ref range = {theta_ref.min():.5f} to {theta_ref.max():.5f}")

fig, ax = plt.subplots(figsize=(5.2, 4.4), dpi=140)
im = ax.contourf(xs, ys, theta_ref, levels=40, cmap="inferno")
ax.set_title("FDM reference theta")
ax.set_xlabel("x*")
ax.set_ylabel("y*")
ax.set_aspect("equal")
fig.colorbar(im, ax=ax)
display(fig)
plt.close(fig)

## 2. Sensor Data and Two-Row Finite-Difference Labels

Only the final 13x2 clean sensor layout is used. The second-row information provides finite-difference labels for the normal temperature gradient near the top boundary.

In [ ]:
sensor_xy = np.array([[x, y] for y in SENSOR_Y for x in SENSOR_X], dtype=float)

def sample_bilinear(xs_array, ys_array, field, points):
    values = []
    for x, y in points:
        i = np.searchsorted(xs_array, x) - 1
        j = np.searchsorted(ys_array, y) - 1
        tx = (x - xs_array[i]) / (xs_array[i + 1] - xs_array[i])
        ty = (y - ys_array[j]) / (ys_array[j + 1] - ys_array[j])
        values.append(
            (1 - tx) * (1 - ty) * field[j, i]
            + tx * (1 - ty) * field[j, i + 1]
            + (1 - tx) * ty * field[j + 1, i]
            + tx * ty * field[j + 1, i + 1]
        )
    return np.asarray(values).reshape(-1, 1)

theta_sensor = sample_bilinear(xs, ys, theta_ref, sensor_xy)

lower, upper, theta_y_labels = [], [], []
for x in SENSOR_X:
    idx = np.where(np.isclose(sensor_xy[:, 0], x))[0]
    idx = idx[np.argsort(sensor_xy[idx, 1])]
    a, b = idx[0], idx[1]
    lower.append(sensor_xy[a])
    upper.append(sensor_xy[b])
    theta_y_labels.append((theta_sensor[b, 0] - theta_sensor[a, 0]) / (sensor_xy[b, 1] - sensor_xy[a, 1]))

lower = np.asarray(lower, dtype=float)
upper = np.asarray(upper, dtype=float)
theta_y_labels = np.asarray(theta_y_labels, dtype=float).reshape(-1, 1)

sensor_xy_t = torch.tensor(sensor_xy, dtype=torch.float64, device=DEVICE)
theta_sensor_t = torch.tensor(theta_sensor, dtype=torch.float64, device=DEVICE)
lower_t = torch.tensor(lower, dtype=torch.float64, device=DEVICE)
upper_t = torch.tensor(upper, dtype=torch.float64, device=DEVICE)
theta_y_t = torch.tensor(theta_y_labels, dtype=torch.float64, device=DEVICE)

fig, ax = plt.subplots(figsize=(7.0, 2.6), dpi=140)
ax.scatter(sensor_xy[:len(SENSOR_X), 0], sensor_xy[:len(SENSOR_X), 1], s=55, color="#f97316", label="lower row")
ax.scatter(sensor_xy[len(SENSOR_X):, 0], sensor_xy[len(SENSOR_X):, 1], s=55, color="#2563eb", label="upper row")
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.955, 1.01)
ax.set_xlabel("x*")
ax.set_ylabel("y*")
ax.set_title("Final 13x2 clean sensor layout")
ax.grid(True, alpha=0.25)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.45), ncol=2)
display(fig)
plt.close(fig)

print(f"sensor theta labels: {len(theta_sensor)}")
print(f"theta_y finite-difference labels: {len(theta_y_labels)}")
print(f"theta_y label range: {theta_y_labels.min():.5f} to {theta_y_labels.max():.5f}")

## 3. PINN and Auxiliary Bi Model

In [ ]:
def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True, retain_graph=True)[0]

def mse(x):
    return torch.mean(x**2)

def sample_lhs(n, seed):
    rng = np.random.default_rng(seed)
    edges = np.linspace(0.0, 1.0, n + 1)
    x = rng.uniform(edges[:-1], edges[1:])
    y = rng.uniform(edges[:-1], edges[1:])
    rng.shuffle(x)
    rng.shuffle(y)
    return torch.tensor(np.c_[x, y], dtype=torch.float64, device=DEVICE).requires_grad_(True)

def edge_points(edge, n):
    s = torch.linspace(0.0, 1.0, n, dtype=torch.float64, device=DEVICE).reshape(-1, 1)
    z, o = torch.zeros_like(s), torch.ones_like(s)
    if edge == "bottom":
        return torch.cat([s, z], dim=1)
    if edge == "top":
        return torch.cat([s, o], dim=1)
    if edge == "left":
        return torch.cat([z, s], dim=1)
    return torch.cat([o, s], dim=1)

class PINN2D(nn.Module):
    def __init__(self, hidden_layers, neurons):
        super().__init__()
        layers = [nn.Linear(2, neurons), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(neurons, neurons), nn.Tanh()]
        layers.append(nn.Linear(neurons, 1))
        self.net = nn.Sequential(*layers)
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, xy):
        return self.net(xy)

class AuxiliaryBiNN(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [nn.Linear(1, AUX_BI_HIDDEN_WIDTH), nn.Tanh()]
        for _ in range(AUX_BI_HIDDEN_LAYERS - 1):
            layers += [nn.Linear(AUX_BI_HIDDEN_WIDTH, AUX_BI_HIDDEN_WIDTH), nn.Tanh()]
        layers.append(nn.Linear(AUX_BI_HIDDEN_WIDTH, 1))
        self.net = nn.Sequential(*layers)
        self.softplus = nn.Softplus()
        init_bi = 0.5 * (H_MIN_GUESS + H_MAX_GUESS) * L / K_S
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.zeros_(layer.bias)
        nn.init.zeros_(self.net[-1].weight)
        nn.init.constant_(self.net[-1].bias, np.log(np.expm1(init_bi)))

    def forward(self, x):
        return self.softplus(self.net(x))


xy_f = sample_lhs(N_DOMAIN, SEED)
bottom_xy = edge_points("bottom", N_BOUNDARY)
top_xy_base = edge_points("top", N_BOUNDARY)
left_xy_base = edge_points("left", N_BOUNDARY)
right_xy_base = edge_points("right", N_BOUNDARY)

pinn = PINN2D(HIDDEN_LAYERS, NEURONS).to(DEVICE)
bi_model = AuxiliaryBiNN().to(DEVICE)
print(pinn)
print(bi_model)

In [ ]:
def total_loss():
    theta_f = pinn(xy_f)
    g = grad(theta_f, xy_f)
    pde = mse(grad(g[:, 0:1], xy_f)[:, 0:1] + grad(g[:, 1:2], xy_f)[:, 1:2] + Q_STAR)

    bottom = mse(pinn(bottom_xy) - 1.0)

    top_xy = top_xy_base.clone().detach().requires_grad_(True)
    theta_top = pinn(top_xy)
    top = mse(-grad(theta_top, top_xy)[:, 1:2] - bi_model(top_xy[:, 0:1]) * theta_top)

    left_xy = left_xy_base.clone().detach().requires_grad_(True)
    right_xy = right_xy_base.clone().detach().requires_grad_(True)
    theta_l, theta_r = pinn(left_xy), pinn(right_xy)
    side = mse(grad(theta_l, left_xy)[:, 0:1] - BI_SIDE * theta_l) + mse(-grad(theta_r, right_xy)[:, 0:1] - BI_SIDE * theta_r)

    data = mse(pinn(sensor_xy_t) - theta_sensor_t)
    theta_y_pred = (pinn(upper_t) - pinn(lower_t)) / (upper_t[:, 1:2] - lower_t[:, 1:2])
    theta_y_data = mse(theta_y_pred - theta_y_t)
    loss = W_PDE * pde + W_BOTTOM * bottom + W_TOP * top + W_SIDE * side + W_DATA * data + W_THETA_Y_DATA * theta_y_data
    return loss, {"pde": pde, "bottom": bottom, "top": top, "side": side, "data": data, "theta_y_data": theta_y_data}

loss0, parts0 = total_loss()
print(f"initial total loss: {float(loss0.detach().cpu()):.4e}")
print({k: float(v.detach().cpu()) for k, v in parts0.items()})

## 4. Train Final Model

In [ ]:
optimizer = torch.optim.Adam([
    {"params": pinn.parameters(), "lr": PINN_LR},
    {"params": bi_model.parameters(), "lr": BI_LR},
])
scheduler = CosineAnnealingLR(optimizer, T_max=N_ADAM, eta_min=ADAM_ETA_MIN)
history = []
t0 = time.perf_counter()

for epoch in range(1, N_ADAM + 1):
    optimizer.zero_grad(set_to_none=True)
    loss, parts = total_loss()
    loss.backward()
    optimizer.step()
    scheduler.step()

    if epoch == 1 or epoch % PRINT_EVERY == 0:
        row = {"epoch": epoch, "total": float(loss.detach().cpu())}
        row.update({k: float(v.detach().cpu()) for k, v in parts.items()})
        history.append(row)
        print(f"Adam {epoch:6d} | L={row['total']:.3e} Top={row['top']:.2e} Data={row['data']:.2e} dTy={row['theta_y_data']:.2e} | {time.perf_counter() - t0:.1f}s")

if RUN_LBFGS:
    lbfgs = torch.optim.LBFGS(list(pinn.parameters()) + list(bi_model.parameters()), lr=1.0, max_iter=20, history_size=50, line_search_fn="strong_wolfe")

    def closure():
        lbfgs.zero_grad(set_to_none=True)
        loss, _ = total_loss()
        loss.backward()
        return loss

    for step in range(1, LBFGS_STEPS + 1):
        lbfgs.step(closure)
        if step == 1 or step % 50 == 0:
            loss, parts = total_loss()
            row = {"epoch": N_ADAM + step, "total": float(loss.detach().cpu())}
            row.update({k: float(v.detach().cpu()) for k, v in parts.items()})
            history.append(row)
            print(f"L-BFGS {step:4d} | L={row['total']:.3e} Top={row['top']:.2e} dTy={row['theta_y_data']:.2e}")

elapsed = time.perf_counter() - t0
print(f"training finished in {elapsed:.1f} s")

## 5. Evaluate and Plot Final Results

In [ ]:
xx, yy = np.meshgrid(xs, ys)
xy_eval = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float64, device=DEVICE)

with torch.no_grad():
    theta_pred = pinn(xy_eval).cpu().numpy().reshape(len(ys), len(xs))
    theta_sensor_pred = pinn(sensor_xy_t).cpu().numpy()
    theta_y_pred = ((pinn(upper_t) - pinn(lower_t)) / (upper_t[:, 1:2] - lower_t[:, 1:2])).cpu().numpy()
    x_plot = np.linspace(0.0, 1.0, 501)[:, None]
    bi_pred = bi_model(torch.tensor(x_plot, dtype=torch.float64, device=DEVICE)).cpu().numpy().ravel()

x_plot = x_plot.ravel()
bi_true = true_bi_numpy(x_plot)
abs_err = np.abs(theta_pred - theta_ref)

theta_rel_l2 = float(np.linalg.norm(theta_pred.ravel() - theta_ref.ravel()) / np.linalg.norm(theta_ref.ravel()))
theta_max_abs = float(abs_err.max())
temp_rmse_C = float(np.sqrt(np.mean((to_temperature(theta_pred) - to_temperature(theta_ref)) ** 2)))
temp_max_abs_C = float(np.max(np.abs(to_temperature(theta_pred) - to_temperature(theta_ref))))
bi_rel_l2 = float(np.linalg.norm(bi_pred - bi_true) / np.linalg.norm(bi_true))

print(f"theta rel L2 : {theta_rel_l2:.4e}")
print(f"theta max abs: {theta_max_abs:.4e}")
print(f"temp RMSE [C]: {temp_rmse_C:.4f}")
print(f"temp max  [C]: {temp_max_abs_C:.4f}")
print(f"Bi rel L2    : {bi_rel_l2:.4e}")

In [ ]:
OUTDIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(6.4, 4.0), dpi=150)
ax.plot(x_plot, bi_true, "k-", lw=2, label="true Bi")
ax.plot(x_plot, bi_pred, "r--", lw=2, label="PINN auxiliary NN")
ax.set_xlabel("x*")
ax.set_ylabel("Bi_top(x*)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(OUTDIR / "bi_profile.png", bbox_inches="tight")
display(fig)
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), dpi=150, constrained_layout=True)
for ax, field, title in zip(axes, [theta_ref, theta_pred, abs_err], ["FDM theta", "PINN theta", "absolute error"]):
    im = ax.contourf(xs, ys, field, levels=40, cmap="inferno" if title != "absolute error" else "viridis")
    ax.scatter(sensor_xy[:, 0], sensor_xy[:, 1], s=20, facecolor="none", edgecolor="cyan")
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlabel("x*")
    ax.set_ylabel("y*")
    fig.colorbar(im, ax=ax)
fig.savefig(OUTDIR / "field_comparison.png", bbox_inches="tight")
display(fig)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.0, 4.0), dpi=150)
for key in ["total", "pde", "top", "data", "theta_y_data"]:
    ax.semilogy([row["epoch"] for row in history], [row[key] for row in history], label=key)
ax.set_xlabel("iteration")
ax.set_ylabel("loss")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
fig.savefig(OUTDIR / "loss_history.png", bbox_inches="tight")
display(fig)
plt.close(fig)

## 6. Energy-Balance Validation

The report uses two physical checks: the local PDE residual and the global heat balance.

In [ ]:
def trapz(values, coordinates):
    return float(np.trapz(values, coordinates))

energy_dir = OUTDIR / "energy_balance_validation"
energy_dir.mkdir(parents=True, exist_ok=True)

dx = float(xs[1] - xs[0])
dy = float(ys[1] - ys[0])
theta_y_grid, theta_x_grid = np.gradient(theta_pred, dy, dx, edge_order=2)
theta_yy = np.gradient(theta_y_grid, dy, axis=0, edge_order=2)
theta_xx = np.gradient(theta_x_grid, dx, axis=1, edge_order=2)
pde_residual = theta_xx + theta_yy + Q_STAR
interior = np.s_[2:-2, 2:-2]

bi_x = np.interp(xs, x_plot, bi_pred)
q_top_cond = -theta_y_grid[-1, :]
q_bottom_cond = theta_y_grid[0, :]
q_left_cond = theta_x_grid[:, 0]
q_right_cond = -theta_x_grid[:, -1]
q_top_robin = bi_x * theta_pred[-1, :]
q_left_robin = BI_SIDE * theta_pred[:, 0]
q_right_robin = BI_SIDE * theta_pred[:, -1]

q_gen = Q_STAR
q_top_cond_int = trapz(q_top_cond, xs)
q_top_robin_int = trapz(q_top_robin, xs)
q_bottom_cond_int = trapz(q_bottom_cond, xs)
q_left_cond_int = trapz(q_left_cond, ys)
q_right_cond_int = trapz(q_right_cond, ys)
q_left_robin_int = trapz(q_left_robin, ys)
q_right_robin_int = trapz(q_right_robin, ys)
q_out_cond = q_top_cond_int + q_bottom_cond_int + q_left_cond_int + q_right_cond_int
q_out_bc = q_top_robin_int + q_bottom_cond_int + q_left_robin_int + q_right_robin_int
imbalance_cond = abs(q_out_cond - q_gen) / abs(q_gen) * 100.0
imbalance_bc = abs(q_out_bc - q_gen) / abs(q_gen) * 100.0

energy_metrics = {
    "grid_nx": int(len(xs)),
    "grid_ny": int(len(ys)),
    "dx": dx,
    "dy": dy,
    "local_residual_metric_nx": int(theta_pred[interior].shape[1]),
    "local_residual_metric_ny": int(theta_pred[interior].shape[0]),
    "boundary_integration_points_top_bottom": int(len(xs)),
    "boundary_integration_points_left_right": int(len(ys)),
    "pde_residual_mean_abs": float(np.mean(np.abs(pde_residual[interior]))),
    "pde_residual_rmse": float(np.sqrt(np.mean(pde_residual[interior] ** 2))),
    "pde_residual_max_abs": float(np.max(np.abs(pde_residual[interior]))),
    "pde_residual_rel_rmse_vs_Q": float(np.sqrt(np.mean(pde_residual[interior] ** 2)) / abs(Q_STAR)),
    "Q_gen_internal": q_gen,
    "Q_out_total_conductive_flux": q_out_cond,
    "Q_out_total_bc_flux": q_out_bc,
    "global_imbalance_conductive_pct": imbalance_cond,
    "global_imbalance_bc_pct": imbalance_bc,
}

with open(energy_dir / "energy_balance_metrics.json", "w") as f:
    json.dump(energy_metrics, f, indent=2)

with open(energy_dir / "global_energy_balance.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["quantity", "value_dimensionless", "description"])
    writer.writerows([
        ("Q_gen_internal", q_gen, "Integral of uniform internal heat generation over unit domain"),
        ("Q_top_cond", q_top_cond_int, "Top outward conductive flux integral, int -theta_y dx"),
        ("Q_top_robin", q_top_robin_int, "Top Robin flux integral, int Bi_NN theta dx"),
        ("Q_bottom_cond", q_bottom_cond_int, "Bottom outward conductive flux integral, int theta_y dx"),
        ("Q_left_cond", q_left_cond_int, "Left outward conductive flux integral, int theta_x dy"),
        ("Q_right_cond", q_right_cond_int, "Right outward conductive flux integral, int -theta_x dy"),
        ("Q_left_robin", q_left_robin_int, "Left side Robin flux integral, int Bi_side theta dy"),
        ("Q_right_robin", q_right_robin_int, "Right side Robin flux integral, int Bi_side theta dy"),
        ("Q_out_total_conductive_flux", q_out_cond, "Total boundary flux from temperature-gradient fluxes"),
        ("Q_out_total_bc_flux", q_out_bc, "Total boundary flux from Robin BC fluxes"),
        ("global_imbalance_conductive_pct", imbalance_cond, "Percent imbalance of conductive boundary flux"),
        ("global_imbalance_bc_pct", imbalance_bc, "Percent imbalance of BC-based boundary flux"),
    ])

print(f"grid: {len(xs)} x {len(ys)}, dx = dy = {dx:.6g}")
print(f"local residual metric grid: {theta_pred[interior].shape[1]} x {theta_pred[interior].shape[0]}")
print(f"PDE residual RMSE / Q*: {100.0 * energy_metrics['pde_residual_rel_rmse_vs_Q']:.4f}%")
print(f"Global conductive imbalance: {imbalance_cond:.5f}%")
print(f"Global BC-based imbalance: {imbalance_bc:.5f}%")

In [ ]:
p98 = float(np.percentile(np.abs(pde_residual[interior]), 98))
levels = np.linspace(-p98, p98, 41)
fig, ax = plt.subplots(figsize=(7.2, 5.4), dpi=150)
cf = ax.contourf(xs, ys, pde_residual, levels=levels, cmap="coolwarm", extend="both")
ax.set_title(r"Local PDE energy residual: $\theta_{xx}+\theta_{yy}+Q^*$")
ax.set_xlabel(r"$x^*$")
ax.set_ylabel(r"$y^*$")
ax.set_aspect("equal")
fig.colorbar(cf, ax=ax, label="residual")
fig.savefig(energy_dir / "local_pde_residual_heatmap.png", bbox_inches="tight")
display(fig)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.4, 4.8), dpi=150)
ax.bar(
    ["Generated heat", "Boundary flux\n(gradient)", "Boundary flux\n(Robin BC)"],
    [q_gen, q_out_cond, q_out_bc],
    color=["#111827", "#2563eb", "#7c3aed"],
)
ax.axhline(q_gen, color="k", ls="--", lw=1.2, label=r"$Q_{gen}$")
ax.set_ylabel("integrated dimensionless heat rate")
ax.set_title("Global energy balance")
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig.savefig(energy_dir / "global_energy_balance_bar.png", bbox_inches="tight")
display(fig)
plt.close(fig)

In [ ]:
with open(energy_dir / "ENERGY_BALANCE_VALIDATION.md", "w") as f:
    f.write("# Energy-Balance Validation\n\n")
    f.write("The final PINN temperature field was checked using the local PDE residual and the global heat balance.\n\n")
    f.write("## Grid Resolution\n\n")
    f.write(f"- Full field grid: {len(xs)} x {len(ys)}\n")
    f.write(f"- Grid spacing: dx = dy = {dx:.6g}\n")
    f.write(f"- Residual metrics: {theta_pred[interior].shape[1]} x {theta_pred[interior].shape[0]} interior points\n")
    f.write(f"- Boundary integration: {len(xs)} points on top/bottom and {len(ys)} points on left/right\n\n")
    f.write("## Local PDE Residual\n\n")
    f.write(r"$$R_{PDE}(x,y)=\theta_{xx}+\theta_{yy}+Q^*$$" + "\n\n")
    f.write(f"- mean |R_PDE|: {energy_metrics['pde_residual_mean_abs']:.6e}\n")
    f.write(f"- RMSE R_PDE: {energy_metrics['pde_residual_rmse']:.6e}\n")
    f.write(f"- RMSE / Q*: {100.0 * energy_metrics['pde_residual_rel_rmse_vs_Q']:.4f}%\n\n")
    f.write("![Local PDE residual](local_pde_residual_heatmap.png)\n\n")
    f.write("## Global Energy Balance\n\n")
    f.write(r"$$Q_{gen}=\int_{\Omega}Q^*dA,\qquad Q_{out}=\int_{\partial\Omega}q_{out}ds$$" + "\n\n")
    f.write(f"- Q_gen: {q_gen:.8f}\n")
    f.write(f"- Q_out from conductive gradients: {q_out_cond:.8f}\n")
    f.write(f"- Q_out from Robin BC fluxes: {q_out_bc:.8f}\n")
    f.write(f"- conductive imbalance: {imbalance_cond:.5f}%\n")
    f.write(f"- BC-based imbalance: {imbalance_bc:.5f}%\n\n")
    f.write("![Global energy balance](global_energy_balance_bar.png)\n")

center_idx = int(np.argmin(np.abs(x_plot - X_JET)))
summary = {
    "run_name": RUN_NAME,
    "reference_source": ref["source"],
    "seed": SEED,
    "sensor_x": SENSOR_X,
    "sensor_y": SENSOR_Y,
    "theta_y_method": "2-row finite difference",
    "weights": {"pde": W_PDE, "bottom": W_BOTTOM, "top": W_TOP, "side": W_SIDE, "data": W_DATA, "theta_y_data": W_THETA_Y_DATA},
    "metrics": {
        "theta_rel_l2": theta_rel_l2,
        "theta_max_abs": theta_max_abs,
        "temp_rmse_C": temp_rmse_C,
        "temp_max_abs_C": temp_max_abs_C,
        "bi_rel_l2": bi_rel_l2,
        "bi_center_true": float(bi_true[center_idx]),
        "bi_center_pred": float(bi_pred[center_idx]),
        "h_center_true": float(bi_true[center_idx] * K_S / L),
        "h_center_pred": float(bi_pred[center_idx] * K_S / L),
        "elapsed_sec": elapsed,
    },
    "energy_balance_metrics": energy_metrics,
}

with open(OUTDIR / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

torch.save({"pinn": pinn.state_dict(), "bi_model": bi_model.state_dict(), "history": history}, OUTDIR / "phase3_auxiliary_nn_pinn.pt")
np.savez_compressed(
    OUTDIR / "phase3_auxiliary_nn_fields.npz",
    xs=xs,
    ys=ys,
    theta_ref=theta_ref,
    theta_pred=theta_pred,
    abs_err=abs_err,
    x_plot=x_plot,
    bi_true=bi_true,
    bi_pred=bi_pred,
    sensor_xy=sensor_xy,
)

print("saved outputs")
print(f"summary: {OUTDIR / 'run_summary.json'}")
print(f"energy balance: {energy_dir}")